╭──────────────────────────────────────────────────────────────────────────────────────────────────╮
│ /                                                                                                │
│ ▲                                                                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
SyntaxError: invalid syntax

In [9]:
import os
from time import gmtime, strftime

import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.image_uris import get_training_image_uri
from sagemaker.core.processing import FrameworkProcessor
from sagemaker.core.shapes import ProcessingOutput, ProcessingS3Output
from sagemaker.train import ModelTrainer
import sagemaker
from sagemaker.train.configs import SourceCode
from sagemaker.core import image_uris


In [10]:
# Configuration
REGION = "us-east-1"
account = 253490779227
ROLE_ARN = (
    "arn:aws:iam::253490779227:role/service-role/AmazonSageMakerAdminIAMExecutionRole"
)

BUCKET = "autonomous-bike"
S3_INPUT_DATA = f"s3://{BUCKET}"


print(f"Role: {ROLE_ARN}")
print(f"Region: {REGION}")
print(f"Default bucket: {BUCKET}")


Role: arn:aws:iam::253490779227:role/service-role/AmazonSageMakerAdminIAMExecutionRole
Region: us-east-1
Default bucket: autonomous-bike


In [11]:
sagemaker_session = Session()

# get_execution_role() only works when your local identity is itself an
# assumed-role (e.g. inside a SageMaker notebook/Studio). Running locally,
# you just pass the ARN of the role SageMaker should assume for the job.
role = ROLE_ARN
print("Using SageMaker execution role:", role)


Using SageMaker execution role: arn:aws:iam::253490779227:role/service-role/AmazonSageMakerAdminIAMExecutionRole


In [12]:
# # Update these paths for your project.


# # Point this at the folder  that contains your SageMaker training entrypoint.
# source_dir = os.path.abspath("requ")
entry_script = "train.py"

# Pick a real SageMaker-supported training image for your region and framework version.
# training_image = get_training_image_uri(
#     region=REGION,
#     framework="pytorch",
#     framework_version="2.8",
#     py_version="py310",
#     instance_type="ml.g5.xlarge",
# )

# training_image


In [ ]:
training_image = f"{account}.dkr.ecr.{REGION}.amazonaws.com/autonomous-bike:latest"

## V3 training

In SageMaker SDK V3, `ModelTrainer` replaces the old framework-specific estimator classes.

You provide:
- a training image
- source code metadata
- structured input channel definitions
- resource config for the training job

In [ ]:
trainer = ModelTrainer(
    training_image=training_image,
    role=ROLE_ARN,
    source_code=SourceCode(
        # source_dir=source_dir,
        entry_script=entry_script,
        requirements_file="requirements.txt"
    ),
)

input_data_config = [
    InputData(channel_name="train", data_source=train_s3_uri),
    InputData(channel_name="val", data_source=val_s3_uri),
]

resource_config = ResourceConfig(
    instance_type="ml.g5.xlarge",
    instance_count=1,
    volume_size_in_gb=100,
)


In [ ]:
# Uncomment to launch training.
# training_job = trainer.train(
#     input_data_config=input_data_config,
#     resource_config=resource_config,
#     output_path=output_s3_uri,
#     hyperparameters={
#         "image-height": "360",
#         "image-width": "640",
#         "backbone": "resnet50",
#         "batch-size": "4",
#         "epochs": "5",
#     },
# )
# training_job


## V3 processing

In SageMaker SDK V3, `FrameworkProcessor` replaces `sagemaker.pytorch.processing.PyTorchProcessor`.

You build it from a framework image URI and then call `.run(...)` with code, source, and outputs.

In [ ]:
processing_image = get_training_image_uri(
    region=region,
    framework="pytorch",
    framework_version="2.1",
    py_version="py310",
    instance_type="ml.m5.xlarge",
)

processor = FrameworkProcessor(
    image_uri=processing_image,
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
)

processing_job_name = "autonomous-bicycle-preprocess-{}".format(
    strftime("%Y-%m-%d-%H-%M-%S", gmtime())
)


In [ ]:
# Uncomment after you add a real preprocessing script.
# processing_job = processor.run(
#     code="preprocessing.py",
#     source_dir=os.path.abspath("scripts/preprocess"),
#     job_name=processing_job_name,
#     outputs=[
#         ProcessingOutput(
#             output_name="train",
#             s3_output=ProcessingS3Output(
#                 s3_uri=f"{processing_output_s3_uri}/train",
#                 local_path="/opt/ml/processing/train",
#                 s3_upload_mode="EndOfJob",
#             ),
#         ),
#         ProcessingOutput(
#             output_name="val",
#             s3_output=ProcessingS3Output(
#                 s3_uri=f"{processing_output_s3_uri}/val",
#                 local_path="/opt/ml/processing/val",
#                 s3_upload_mode="EndOfJob",
#             ),
#         ),
#     ],
#     wait=False,
# )
# processing_job


## Notes

- Your original notebook used V2 imports. In V3, the replacement is `ModelTrainer` and `FrameworkProcessor`.
- The old `from sagemaker.pytorch import PyTorch` path should not be used in a V3 migration.
- If `get_execution_role()` fails locally, set `role = "arn:aws:iam::<account-id>:role/<your-role>"` yourself.
- Your current notebook kernel is Python 3.13. SageMaker V3 docs list Python 3.9+ generally, but in practice Python 3.10 or 3.11 is the safer choice if you hit package issues.